In [1]:
import os, sys, json, torch
from collections import Counter
from tqdm.auto import tqdm

sys.path.insert(0, os.path.abspath(".."))

from transformers import AutoTokenizer
from data.pt_dataset import get_dataset
from train_sorl_post import load_checkpoint

# ── Config ──────────────────────────────────────────────────
CKPT_DIR    = "/Users/fangyuanyu/Implementation/mod_gpt/ckpt/06b-128v"
MODEL_NAME  = "Qwen/Qwen3-0.6B"
DATASET     = "scienceqa"
ABS_VOCAB   = 128          # C_SIZE (128 codes + 1 placeholder = 129 rows)
K           = 4            # abstract token every K NL tokens
MAX_NEW     = 256
EVAL_BATCH  = 32
DEVICE      = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Device: {DEVICE}")

/Users/fangyuanyu/anaconda3/lib/python3.11/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


Device: mps


In [2]:
# ── Load model + dataset ────────────────────────────────────
model, tokenizer, base_vocab = load_checkpoint(
    MODEL_NAME, ABS_VOCAB, CKPT_DIR, DEVICE,
    untie_embeddings=False, separate_abs_params=False,
)
model.eval()

pad_id = tokenizer.pad_token_id or 0
test_ds = get_dataset(DATASET, split="test", tokenizer=tokenizer, max_length=512)
extract_fn = test_ds.extract_answer
N = len(test_ds)
print(f"Loaded {N} test samples  |  base_vocab={base_vocab}  |  total_vocab={model.total_vocab_size.item()}")

Loading base model: Qwen/Qwen3-0.6B


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Loading full model weights from: /Users/fangyuanyu/Implementation/mod_gpt/ckpt/06b-128v/model.safetensors
  Loaded 315 tensors (missing=1, unexpected=0)
  Missing keys (first 5): ['model.lm_head.weight']
Loading abstract embeddings from: /Users/fangyuanyu/Implementation/mod_gpt/ckpt/06b-128v/abs_embeddings.pt
  Restored abstract rows: embed=torch.Size([129, 1024]), lm_head=torch.Size([129, 1024])
  Tied ckpt: True | untie_embeddings=False
  Step: 3737, Epoch: 1
Loaded 2224 test samples  |  base_vocab=151936  |  total_vocab=152065


In [3]:
def left_pad_and_mask(prompts, pad_id):
    """Left-pad a list of 1-D tensors and return (input_ids, attention_mask)."""
    max_len = max(p.size(0) for p in prompts)
    input_ids, masks = [], []
    for p in prompts:
        pad_len = max_len - p.size(0)
        input_ids.append(torch.cat([torch.full((pad_len,), pad_id, dtype=p.dtype), p]))
        masks.append(torch.cat([torch.zeros(pad_len, dtype=torch.long), torch.ones(p.size(0), dtype=torch.long)]))
    return torch.stack(input_ids), torch.stack(masks)


def decode_interleaved(token_ids, tokenizer, base_vocab):
    """Decode a token-id sequence, inserting ⟨ABS_N⟩ markers for abstract tokens."""
    parts, nl_buf = [], []
    for tid in token_ids:
        if tid < base_vocab:
            nl_buf.append(tid)
        else:
            if nl_buf:
                parts.append(tokenizer.decode(nl_buf, skip_special_tokens=True))
                nl_buf = []
            parts.append(f"⟨ABS_{tid - base_vocab}⟩")
    if nl_buf:
        parts.append(tokenizer.decode(nl_buf, skip_special_tokens=True))
    return "".join(parts)


print("Helpers defined.")

Helpers defined.


In [ ]:
# ── Batched generation over full test set ───────────────────
records = []           # per-sample dicts
global_abs_counter = Counter()
correct_total = 0

for bs_start in tqdm(range(0, N, EVAL_BATCH), desc="eval"):
    bs_end = min(bs_start + EVAL_BATCH, N)

    prompts, prompt_lens = [], []
    for i in range(bs_start, bs_end):
        sample = test_ds[i]
        pl = sample["prompt_len"]
        prompts.append(sample["input_ids"][:pl])
        prompt_lens.append(pl)

    input_ids, attn_mask = left_pad_and_mask(prompts, pad_id)
    input_ids, attn_mask = input_ids.to(DEVICE), attn_mask.to(DEVICE)

    with torch.no_grad():
        generated = model.generate(
            input_ids=input_ids,
            attention_mask=attn_mask,
            max_new_tokens=MAX_NEW,
            temperature=0.0,
            K=K,
        )

    max_pl = input_ids.size(1)
    for j, i in enumerate(range(bs_start, bs_end)):
        pad_len = max_pl - prompt_lens[j]
        gen_ids  = generated[j, pad_len:]          # prompt + generated
        new_ids  = generated[j, max_pl:]            # only new tokens

        # ── abstract-token stats ──
        new_ids_list = new_ids.cpu().tolist()
        abs_mask = [tid >= base_vocab for tid in new_ids_list]
        abs_ids  = [tid - base_vocab for tid, m in zip(new_ids_list, abs_mask) if m]
        n_abs    = len(abs_ids)
        n_nl     = len(new_ids_list) - n_abs
        for aid in abs_ids:
            global_abs_counter[aid] += 1

        # ── decode ──
        nl_ids       = gen_ids[gen_ids < base_vocab].cpu().tolist()
        full_text    = tokenizer.decode(nl_ids, skip_special_tokens=True)
        prompt_text  = tokenizer.decode(prompts[j].tolist(), skip_special_tokens=True)
        interleaved  = decode_interleaved(new_ids_list, tokenizer, base_vocab)

        # ── gold / pred ──
        ref_sample   = test_ds[i]
        ref_ids      = ref_sample["input_ids"][ref_sample["input_ids"] < base_vocab]
        ref_text     = tokenizer.decode(ref_ids.tolist(), skip_special_tokens=True)
        pred         = extract_fn(full_text)
        gold         = extract_fn(ref_text)
        hit          = pred is not None and gold is not None and pred.strip() == gold.strip()
        correct_total += int(hit)

        records.append({
            "idx": i,
            "question": prompt_text,
            "response_nl": full_text[len(prompt_text):].strip(),
            "interleaved": interleaved,
            "pred": pred,
            "gold": gold,
            "correct": hit,
            "abs_ids": abs_ids,
            "n_abs": n_abs,
            "n_nl": n_nl,
        })

acc = correct_total / max(N, 1)
print(f"\\nAccuracy: {correct_total}/{N} = {acc*100:.1f}%")

eval:   0%|          | 0/70 [00:00<?, ?it/s]

In [ ]:
# ── Global inner-monologue statistics ───────────────────────
total_abs = sum(r["n_abs"] for r in records)
total_nl  = sum(r["n_nl"]  for r in records)
effective_vocab = len(global_abs_counter)

print(f"Effective abstract vocab: {effective_vocab} / {ABS_VOCAB}")
print(f"Total abstract tokens:   {total_abs}")
print(f"Total NL tokens:         {total_nl}")
print(f"Abstract ratio:          {total_abs / max(total_abs + total_nl, 1):.2%}")
print(f"\\nTop-20 abstract codes:")
for code, cnt in global_abs_counter.most_common(20):
    print(f"  ABS_{code:>3d}: {cnt:>5d} ({cnt/total_abs:.1%})")

In [ ]:
# ── Print first 10 samples ──────────────────────────────────
for r in records[:10]:
    mark = "✓" if r["correct"] else "✗"
    print(f"\\n{'='*80}")
    print(f"[{r['idx']}] {mark}  gold={r['gold']}  pred={r['pred']}  abs={r['n_abs']}  nl={r['n_nl']}")
    print(f"Q: {r['question'][:200]}")
    print(f"NL response: {r['response_nl'][:300]}")
    print(f"Interleaved: {r['interleaved'][:500]}")
    print(f"Abs IDs: {r['abs_ids'][:30]}")

In [ ]:
# ── Save full log to JSON ───────────────────────────────────
out_path = os.path.join(CKPT_DIR, "inner_monologue_sciqa.json")
payload = {
    "ckpt": CKPT_DIR,
    "dataset": DATASET,
    "K": K,
    "accuracy": acc,
    "correct": correct_total,
    "total": N,
    "effective_vocab": effective_vocab,
    "total_abs": total_abs,
    "total_nl": total_nl,
    "abs_ratio": total_abs / max(total_abs + total_nl, 1),
    "top20": global_abs_counter.most_common(20),
    "freq_distribution": dict(global_abs_counter),
    "samples": records,
}
with open(out_path, "w") as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)
print(f"Saved {len(records)} records → {out_path}")

In [ ]:
# ── Correct vs Incorrect breakdown ──────────────────────────
import numpy as np

correct_recs = [r for r in records if r["correct"]]
wrong_recs   = [r for r in records if not r["correct"]]

def abs_stats(recs):
    if not recs:
        return {}
    n_abs = [r["n_abs"] for r in recs]
    n_nl  = [r["n_nl"]  for r in recs]
    # unique abs codes used per sample
    uniq  = [len(set(r["abs_ids"])) for r in recs]
    return {
        "count": len(recs),
        "avg_abs": np.mean(n_abs),
        "avg_nl":  np.mean(n_nl),
        "avg_unique_abs": np.mean(uniq),
        "avg_abs_ratio": np.mean([a/(a+n+1e-9) for a,n in zip(n_abs, n_nl)]),
    }

print("Correct samples:")
for k,v in abs_stats(correct_recs).items():
    print(f"  {k}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")

print("\\nWrong samples:")
for k,v in abs_stats(wrong_recs).items():
    print(f"  {k}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")

In [ ]:
# ── Abstract code frequency histogram ───────────────────────
import matplotlib.pyplot as plt

codes = sorted(global_abs_counter.keys())
freqs = [global_abs_counter[c] for c in codes]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Rank-frequency (Zipf-like)
ranked = sorted(freqs, reverse=True)
axes[0].bar(range(len(ranked)), ranked, color="steelblue", width=1.0)
axes[0].set_xlabel("Rank")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Abstract Code Frequency (rank-ordered)")

# Code ID histogram
axes[1].bar(codes, freqs, color="coral", width=1.0)
axes[1].set_xlabel("Abstract Code ID")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Frequency by Code ID")

plt.tight_layout()
plt.savefig(os.path.join(CKPT_DIR, "abs_code_freq_sciqa.png"), dpi=150)
plt.show()
print("Saved figure.")